<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 04. PCA: Encontrar los Ejes Ocultos de tus Datos
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Programación para Ciencia de Datos
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 06
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Data%20Science%20programming/06%20-%20Feature%20Engineering/Para%20Dummies/04_PCA_Feature_Engineering_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **"para no ingenieros"** del módulo 04 de Feature Engineering. En el cuaderno anterior creamos columnas nuevas "a mano" (razones, logaritmos, conteos). Ahora vamos a conocer una técnica que crea columnas nuevas **automáticamente**: el **Análisis de Componentes Principales**, o **PCA** por sus siglas en inglés.

Al terminar podrás explicar, con tus propias palabras:
1. Qué problema resuelve PCA y por qué se dice que "encuentra ejes ocultos".
2. Qué son las **cargas (loadings)** y cómo ayudan a interpretar cada componente.
3. Qué es la **varianza explicada** y por qué no es lo mismo que "poder predictivo".
4. Cómo usar `PCA` de scikit-learn con datos reales, paso a paso.

---
## 1. El problema: dos columnas que "se mueven juntas" 📏

Piensa en la **estatura** y la **envergadura** (la distancia entre las puntas de los dedos con los brazos extendidos) de un grupo de personas. Ambas miden, en el fondo, algo parecido: **qué tan grande es la persona**. Si conoces la estatura de alguien, ya puedes adivinar razonablemente bien su envergadura, porque las dos "se mueven juntas".

Cuando dos (o más) columnas están fuertemente correlacionadas así, en realidad estamos **pagando dos veces por la misma información**. PCA se pregunta: *¿existe una manera más inteligente de describir a estas personas, usando ejes que no sean exactamente "estatura" y "envergadura", sino combinaciones de ambas?*

La respuesta de PCA son dos nuevos ejes:
1. Un eje de **"Tamaño"**: mezcla estatura y envergadura por igual, para capturar qué tan grande es la persona en general.
2. Un eje de **"Forma"**: contrasta estatura contra envergadura, para capturar si una persona es relativamente "larga de brazos" o "compacta" en comparación con su estatura.

> 📌 **Para recordar:** PCA no inventa información nueva — **rota** los ejes originales para alinearlos con las direcciones en las que los datos realmente varían.

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)
n = 12

# Generamos estaturas variadas y una envergadura correlacionada (con algo de ruido)
estatura_cm = np.random.normal(170, 12, n)
envergadura_cm = estatura_cm + np.random.normal(0, 4, n)

personas = pd.DataFrame({
    "persona": [f"Persona {i + 1}" for i in range(n)],
    "estatura_cm": estatura_cm.round(1),
    "envergadura_cm": envergadura_cm.round(1),
})
personas

### 🤔 ¿Qué acaba de pasar?

- Generamos estaturas al azar alrededor de 170 cm, y construimos la envergadura como "la estatura, más un poco de ruido" — así garantizamos que ambas columnas estén fuertemente correlacionadas, tal como sucede con datos reales de este tipo.
- Si dibujaras estos puntos en un plano (estatura contra envergadura), notarías que no se dispersan en cualquier dirección: forman una especie de "nube alargada en diagonal". Esa dirección diagonal es justamente el eje de **"Tamaño"** que PCA va a encontrar.

---
## 2. Paso obligatorio antes de PCA: estandarizar 📐

PCA compara qué tanto "varía" cada columna. Si una columna estuviera en centímetros y otra en kilómetros, la de kilómetros parecería "variar menos" solo por la escala, no porque en realidad contenga menos información. Para evitar ese sesgo, **siempre estandarizamos** las columnas antes de aplicar PCA: le restamos su promedio y la dividimos entre su desviación estándar, para que todas queden en una escala comparable (promedio 0, desviación estándar 1).

In [ ]:
from sklearn.decomposition import PCA

X = personas[["estatura_cm", "envergadura_cm"]]

# Estandarizacion manual: (valor - promedio) / desviacion estandar
X_estandarizado = (X - X.mean()) / X.std()
X_estandarizado.round(2).head()

### 🤔 ¿Qué acaba de pasar?

- `X.mean()` y `X.std()` calculan el promedio y la desviación estándar de cada columna por separado.
- Al restar el promedio y dividir entre la desviación estándar, ambas columnas quedan "en las mismas unidades": ya no importa si originalmente estaban en centímetros, pulgadas o cualquier otra escala. Ahora un valor de `1.0` significa lo mismo en ambas columnas: "una desviación estándar por encima del promedio".

---
## 3. Ajustando PCA y leyendo las "cargas" (loadings) ⚖️

Ya con los datos estandarizados, podemos ajustar `PCA` de scikit-learn. Lo interesante no es solo el algoritmo, sino sus **cargas (loadings)**: los "pesos" que indican cómo se construye cada componente a partir de las columnas originales.

In [ ]:
pca = PCA()
pca.fit(X_estandarizado)

nombres_componentes = ["PC1 (Tamano)", "PC2 (Forma)"]

cargas = pd.DataFrame(
    pca.components_.T,
    columns=nombres_componentes,
    index=X.columns
)
cargas.round(3)

### 🤔 ¿Qué acaba de pasar?

- `pca.fit(X_estandarizado)` le muestra a PCA los datos y le pide que encuentre los ejes de máxima variación.
- `pca.components_` guarda las cargas; lo transponemos (`.T`) para que cada fila sea una columna original y cada columna sea un componente principal.
- En **PC1**, `estatura_cm` y `envergadura_cm` deberían tener cargas del **mismo signo** y magnitud parecida (ambas alrededor de $\pm 0.707$): esto confirma que PC1 es el eje de **"Tamaño"**, donde ambas variables "suman" en la misma dirección.
- En **PC2**, las cargas deberían tener **signos opuestos**: ese es el eje de **"Forma"**, que contrasta a alguien relativamente "más envergadura que estatura" contra alguien "más estatura que envergadura".

---
## 4. Varianza explicada: ¿cuánta información captura cada eje? 📊

No todos los componentes son igual de importantes. `explained_variance_ratio_` nos dice qué porcentaje de la variación total de los datos originales queda capturado por cada componente.

In [ ]:
import matplotlib.pyplot as plt

varianza = pca.explained_variance_ratio_

for nombre, v in zip(nombres_componentes, varianza):
    print(f"{nombre}: {v * 100:.1f}% de la varianza total")

plt.figure(figsize=(5, 3.5))
plt.bar(nombres_componentes, varianza * 100, color=["#f59e0b", "#fcd34d"])
plt.ylabel("% de varianza explicada")
plt.title("¿Cuanta informacion aporta cada componente?")
plt.ylim(0, 100)
plt.tight_layout()
plt.show()

### 🤔 ¿Qué acaba de pasar?

- Como `estatura_cm` y `envergadura_cm` están muy correlacionadas, casi toda la información se concentra en **PC1 ("Tamaño")** — normalmente vas a ver un porcentaje bastante alto ahí (piensa en un 90% o más), y el resto le queda a **PC2 ("Forma")**.
- Esto significa que, si tuvieras que quedarte con una sola columna en lugar de dos, `PC1` por sí solo ya resume la mayor parte de la información original: esa es la idea de **reducción de dimensionalidad**.

> ⚠️ **Ojo:** que un componente explique mucha varianza **no** significa automáticamente que sea el más útil para predecir tu variable objetivo. A veces un componente "pequeño" (como PC2, "Forma") resulta sorprendentemente informativo para una pregunta específica.

---
## 5. Usar los componentes como columnas nuevas 🧩

Finalmente, transformamos los datos originales a las nuevas coordenadas (`PC1`, `PC2`). Estas columnas ya están **descorrelacionadas entre sí** — a diferencia de `estatura_cm` y `envergadura_cm`, que iban casi siempre de la mano.

In [ ]:
X_pca = pca.transform(X_estandarizado)
X_pca = pd.DataFrame(X_pca, columns=nombres_componentes)

resultado = pd.concat([personas[["persona"]], X_pca.round(2)], axis=1)
resultado

### 🤔 ¿Qué acaba de pasar?

- `pca.transform(...)` aplica la misma rotación que PCA aprendió, y nos devuelve las coordenadas de cada persona en los nuevos ejes.
- Ahora cada persona tiene un valor de `PC1 (Tamano)` (qué tan grande es en general) y `PC2 (Forma)` (si es relativamente más "envergadura" o más "estatura"). Estas dos columnas nuevas podrían alimentar directamente un modelo, en lugar de las dos columnas originales correlacionadas.

---
## 6. Buenas prácticas antes de usar PCA ✅

| Recomendación | Por qué importa |
|---|---|
| Usa solo columnas numéricas | PCA se basa en operaciones matemáticas; no funciona directamente con texto o categorías. |
| Estandariza siempre (salvo que todo esté ya en la misma escala) | Evita que una columna "domine" solo por tener números más grandes. |
| Revisa los valores atípicos primero | Un solo dato extremo puede distorsionar los ejes que PCA encuentra. |
| No confundas varianza explicada con poder predictivo | Un componente puede ser "pequeño" en varianza y aun así muy útil para tu pregunta específica. |

---
## 7. Resumen relámpago ⚡

| Idea | En una frase |
|---|---|
| PCA | Encuentra ejes nuevos (rotados) que resumen mejor cómo varían tus datos originales. |
| Estandarizar | Paso obligatorio antes de PCA, para que ninguna columna domine solo por su escala. |
| Cargas (loadings) | Los "pesos" que muestran cómo se construye cada componente a partir de las columnas originales. |
| Varianza explicada | Qué porcentaje de la variación total captura cada componente — no es lo mismo que "poder predictivo". |
| Componentes como columnas nuevas | `PC1`, `PC2`, ... pueden usarse directamente como características, ya descorrelacionadas entre sí. |

➡️ **Siguiente paso:** en el cuaderno [05 - Selección de Características e Información Mutua (Para Dummies)](05_Seleccion_Caracteristicas_y_Mutual_Information_Dummies.ipynb) aprenderás a **elegir** cuáles columnas vale la pena conservar, en lugar de combinarlas como hicimos aquí con PCA.

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Programación para Ciencia de Datos (Edición Para No Ingenieros)</i>
  </p>
</div>